In [ ]:
import sys, os, json, traceback
from pathlib import Path

KAGGLE_WORKING = Path('/kaggle/working')
REPORT = KAGGLE_WORKING / 'smoke-report.txt'
lines = []
def log(msg):
    lines.append(msg)
    print(msg)

log(f'Python: {sys.version}')
log(f'CWD: {os.getcwd()}')

# 1. CUDA check
try:
    import torch
    log(f'CUDA available: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        log(f'GPU: {torch.cuda.get_device_name(0)}')
        log(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f}GB')
except Exception:
    log(f'torch import FAILED: {traceback.format_exc()}')

# 2. sentence-transformers check
try:
    from sentence_transformers import SentenceTransformer
    log('sentence_transformers import: OK')
except Exception:
    log(f'sentence_transformers import FAILED: {traceback.format_exc()}')

# 3. Repo check
try:
    REPO_DIR = Path.cwd()
    if not (REPO_DIR / 'training' / 'train.py').exists():
        matches = list(KAGGLE_WORKING.glob('*/training/train.py'))
        if matches:
            REPO_DIR = matches[0].parents[1]
        else:
            clone_dir = KAGGLE_WORKING / 'Modern-AI-Log-Filter-Training'
            if not clone_dir.exists():
                import subprocess
                subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/Jacob-Valor/Modern-AI-Log-Filter-Training.git',
                    str(clone_dir)], check=True)
            REPO_DIR = clone_dir
    log(f'REPO_DIR: {REPO_DIR}')
    log(f'train.py exists: {(REPO_DIR / "training" / "train.py").exists()}')
except Exception:
    log(f'Repo locate FAILED: {traceback.format_exc()}')

# 4. Data check
try:
    PREPROCESSED_DIR = REPO_DIR / 'HDFS_v3_TraceBench' / 'preprocessed'
    log(f'PREPROCESSED_DIR: {PREPROCESSED_DIR}')
    log(f'normal_trace.csv exists: {(PREPROCESSED_DIR / "normal_trace.csv").exists()}')
    log(f'failure_trace.csv exists: {(PREPROCESSED_DIR / "failure_trace.csv").exists()}')
    if (PREPROCESSED_DIR / 'normal_trace.csv').exists():
        import pandas as pd
        df = pd.read_csv(PREPROCESSED_DIR / 'normal_trace.csv', nrows=5)
        log(f'normal_trace.csv shape (5 rows): {df.shape}')
        log(f'columns: {list(df.columns[:10])}...')
except Exception:
    log(f'Data check FAILED: {traceback.format_exc()}')

# 5. text_dataset import check
try:
    sys.path.insert(0, str(REPO_DIR))
    sys.path.insert(0, str(REPO_DIR / 'src'))
    from training.text_dataset import build_windows
    log('build_windows import: OK')
except Exception:
    log(f'build_windows import FAILED: {traceback.format_exc()}')

REPORT.write_text('\n'.join(lines))
print(f'\nReport: {REPORT}')
print('\n'.join(lines))